# LoRA-corrected teacher eval: general knowledge

Part of plan.md step 3.3 -- same suite as 1.2's baseline eval and
`10c_teacher_corrected_general_knowledge.ipynb`'s run on the
full-parameter corrected teacher: MMLU (5-shot), ARC-Challenge (25-shot),
Winogrande (5-shot), full test sets, shot counts pinned in
`bonsai.eval.general_knowledge.NUM_FEWSHOT`. Only the model pointer
changes, to the LoRA-corrected teacher (`sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora`, 3.3).

Comparison point: full-parameter corrected teacher showed no meaningful
regression here (all deltas within ~1pp of the base model) despite a
real HumanEval-TS regression -- this checks whether LoRA correction
preserves general knowledge the same way.

In [1]:
import json

from bonsai.config import load_config
from bonsai.eval.general_knowledge import run_general_knowledge
from bonsai.tracking import log_metric, new_run, save_config_snapshot

config = load_config("../configs/qwen3-30b-a3b.toml")
model_repo_id = "sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora"
run_dir = new_run("teacher-corrected-lora-eval")
save_config_snapshot(run_dir, config)

print(f"model: {model_repo_id}")
print(f"run dir: {run_dir}")

model: sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected-lora
run dir: runs/20260925T044501Z-73bdd7-teacher-corrected-lora-eval


In [2]:
general_knowledge = run_general_knowledge(
    model_repo_id, revision="main", dtype="bfloat16", batch_size="auto"
)

for task, result in general_knowledge.items():
    print(task, result["results"][task])

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/1.70M [00:00<?, ?B/s]

... (5,568 outputs left out)

Running loglikelihood requests:  96%|█████████▌| 2433/2534 [00:27<00:00, 116.20it/s]

Running loglikelihood requests:  99%|█████████▊| 2497/2534 [00:27<00:00, 130.02it/s]

Running loglikelihood requests: 100%|██████████| 2534/2534 [00:27<00:00, 91.21it/s] 

mmlu {'alias': 'mmlu', 'name': 'mmlu', 'sample_len': 14042, 'acc,none': 0.8070075487822248, 'acc_stderr,none': 0.003159025228294166, 'sample_count': {'acc,none': 14042}}
arc_challenge {'name': 'arc_challenge', 'alias': 'arc_challenge', 'sample_len': 1172, 'acc,none': 0.6484641638225256, 'acc_stderr,none': 0.013952413699601044, 'acc_norm,none': 0.6715017064846417, 'acc_norm_stderr,none': 0.013724978465537458}
winogrande {'name': 'winogrande', 'alias': 'winogrande', 'sample_len': 1267, 'acc,none': 0.7845303867403315, 'acc_stderr,none': 0.011555295286059255}


In [3]:
(run_dir / "general-knowledge.json").write_text(
    json.dumps(general_knowledge, indent=2, default=str)
)

for task, result in general_knowledge.items():
    log_metric(run_dir, task=task, **result["results"][task])

## Comparison

| task | base | full-parameter corrected | LoRA-corrected |
|---|---|---|---|
| MMLU (5-shot) | 0.8101 | 0.8082 | see above |
| ARC-Challenge acc | 0.6536 | 0.6433 | see above |
| ARC-Challenge acc_norm | 0.6749 | 0.6689 | see above |
| Winogrande (5-shot) | 0.7798 | 0.7806 | see above |